# RF-DETR Playground — Interactive Exploration

This notebook walks through the full workflow:
1. Download a dataset
2. Explore the data
3. Train an RF-DETR model
4. Evaluate and visualize results

In [ ]:
# Install dependencies (run once)
# !pip install -e ".[all]"

## 1. Download Dataset

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), ''))

from rf_detr_playground.data.downloader import download_dataset
from rf_detr_playground.data.registry import list_datasets, get_dataset

print("Available datasets:", list_datasets())

# Download aquarium (small, fast)
dataset_path = download_dataset("aquarium")
print(f"Dataset ready at: {dataset_path}")

## 2. Explore the Dataset

In [ ]:
import json
from pathlib import Path

# Load annotations
ann_path = Path("data/aquarium/train/_annotations.coco.json")
with open(ann_path) as f:
    coco = json.load(f)

print(f"Images: {len(coco['images'])}")
print(f"Annotations: {len(coco['annotations'])}")
print(f"Categories: {[c['name'] for c in coco['categories']]}")

# Class distribution
from collections import Counter
class_counts = Counter()
cat_map = {c['id']: c['name'] for c in coco['categories']}
for ann in coco['annotations']:
    class_counts[cat_map[ann['category_id']]] += 1

print("\nClass distribution:")
for cls, count in class_counts.most_common():
    print(f"  {cls}: {count}")

In [ ]:
# Visualize sample images
from rf_detr_playground.visualization.detections import visualize_dataset_sample

visualize_dataset_sample("data/aquarium", num_images=6)

## 3. Train a Model

In [ ]:
from rf_detr_playground.training.trainer import TrainingConfig, train

config = TrainingConfig(
    dataset_name="aquarium",
    model_size="base",
    epochs=10,       # Increase for better results
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    run_name="aquarium_demo",
)

output_dir = train(config)

## 4. Evaluate the Model

In [ ]:
from rf_detr_playground.evaluation.evaluator import evaluate_model

results = evaluate_model(
    checkpoint_path=str(output_dir / "best.pt"),
    dataset_dir="data/aquarium",
    model_size="base",
    output_dir="outputs/eval_aquarium_demo",
)

In [ ]:
# Visualize per-class AP
from rf_detr_playground.visualization.plots import plot_per_class_ap

if results.get("per_class_ap"):
    plot_per_class_ap(
        results["per_class_ap"],
        Path("outputs/eval_aquarium_demo/per_class_ap.png"),
    )

## 5. Run Inference on Sample Images

In [ ]:
import matplotlib.pyplot as plt
from rf_detr_playground.training.trainer import get_model
from rf_detr_playground.visualization.detections import draw_detections

# Load model
model = get_model("base")

# Pick a sample image
sample_img = list(Path("data/aquarium/valid").glob("*.jpg"))[0]
detections = model.predict(str(sample_img), threshold=0.5)

# Visualize
info = get_dataset("aquarium")
annotated = draw_detections(sample_img, detections, class_names=info.classes)

plt.figure(figsize=(12, 8))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"Detections: {len(detections)} objects")
plt.show()